In [3]:
import sqlite3
import pandas as pd

con = sqlite3.connect(r'C:\Users\MUEENUDHEEN\OneDrive\Desktop\airtel-telecom-analytics\data\airtel.db') 

def q(sql):
    return pd.read_sql(sql, con)

In [36]:
q("""
SELECT
    COUNT(*) AS subs,
    SUM(is_churned) AS left_
FROM customer_features
""")

DatabaseError: Execution failed on sql '
SELECT
    COUNT(*) AS subs,
    SUM(is_churned) AS left_
FROM customer_features
': no such table: customer_features

cell 1
The numbers — one row per subscriber
1. For each subscriber, average their daily data across the four months before the offer.
2. That gives one number per subscriber — the correct grain, because subscribers were what was randomised.
3. Pull it into a DataFrame called pre with q().

In [5]:
q("""
SELECT
    c.ab_group AS g,
    u.customer_id,
    AVG(u.data_mb) AS m
FROM usage_daily u
JOIN customer_features c ON c.customer_id = u.customer_id
WHERE u.usage_date < '2025-05-01'
GROUP BY c.ab_group, u.customer_id
""")

"\nSELECT\n    c.ab_group AS g,\n    u.customer_id,\n    AVG(u.data_mb) AS m\nFROM usage_daily u\nJOIN customer_features c ON c.customer_id = u.customer_id\nWHERE u.usage_date < '2025-05-01'\nGROUP BY c.ab_group, u.customer_id\n"

cell 2
The identical check at the wrong grain
1. Run the same comparison, on the same period, without the GROUP BY customer_id.
2. That is one row per usage day rather than per subscriber — 912,820 rows instead of 21,993.
3. Run the identical F-test and Z-test on the rows.
4. Compare the p-value with the one from T1

In [6]:
q("""
SELECT c.ab_group AS g, u.data_mb AS m -- one row per DAY
FROM usage_daily u -- one row = a day
JOIN customer_features c ON c.customer_id = u.customer_id -- arm
WHERE u.usage_date < '2025-05-01'; -- same period
""")

"\nSELECT c.ab_group AS g, u.data_mb AS m -- one row per DAY\nFROM usage_daily u -- one row = a day\nJOIN customer_features c ON c.customer_id = u.customer_id -- arm\nWHERE u.usage_date < '2025-05-01'; -- same period\n"

cell 3
the six remaining balance checks
1. Test four averages fixed before May: stated ARPU, tenure, complaint count, and revenue from January to
April. For each one, F-test first, then the road from 4.10.
2. Test two percentages: smartphone share and prepaid share, with z_prop2.
3. All six are two-tailed. Record every result, including the ones that do not come out clean.

In [7]:
q("""
SELECT c.ab_group AS g, -- the arm
 COALESCE(SUM(r.amount_inr), 0) AS s -- paid, Jan-Apr
FROM customer_features c -- every sub
LEFT JOIN recharges r ON r.customer_id = c.customer_id -- keep all
 AND r.recharge_date >= '2025-01-01' -- from January
 AND r.recharge_date < '2025-05-01' -- to the offer
GROUP BY c.customer_id; -- one row per sub
""")

"\nSELECT c.ab_group AS g, -- the arm\n COALESCE(SUM(r.amount_inr), 0) AS s -- paid, Jan-Apr\nFROM customer_features c -- every sub\nLEFT JOIN recharges r ON r.customer_id = c.customer_id -- keep all\n AND r.recharge_date >= '2025-01-01' -- from January\n AND r.recharge_date < '2025-05-01' -- to the offer\nGROUP BY c.customer_id; -- one row per sub\n"

In [8]:
q("""
SELECT ab_group, COUNT(*) AS n, -- the arm
 SUM(device_type = 'Smartphone') AS smart, -- smartphones
 SUM(connection_type = 'Prepaid') AS prepaid -- prepaid
FROM customer_features GROUP BY ab_group; -- two rows
""")

"\nSELECT ab_group, COUNT(*) AS n, -- the arm\n SUM(device_type = 'Smartphone') AS smart, -- smartphones\n SUM(connection_type = 'Prepaid') AS prepaid -- prepaid\nFROM customer_features GROUP BY ab_group; -- two rows\n"

The numbers — the same grain as T1, the window instead of the months before it

In [9]:
q("""
SELECT c.ab_group AS g, u.customer_id, -- one row per sub
 AVG(u.data_mb) AS m -- their own mean
FROM usage_daily u -- one row = a day
JOIN customer_features c ON c.customer_id = u.customer_id -- arm
WHERE u.usage_date >= '2025-05-01' -- the offer window
GROUP BY u.customer_id; -- the right grain
""")

"\nSELECT c.ab_group AS g, u.customer_id, -- one row per sub\n AVG(u.data_mb) AS m -- their own mean\nFROM usage_daily u -- one row = a day\nJOIN customer_features c ON c.customer_id = u.customer_id -- arm\nWHERE u.usage_date >= '2025-05-01' -- the offer window\nGROUP BY u.customer_id; -- the right grain\n"

The numbers — two counts, and a second count that only includes leavers from 1 May
Churn before the offer started cannot have been caused by it, so the fairer test counts only those who left on or
after 2025-05-01.

In [10]:
("""
SELECT ab_group, COUNT(*) AS n, -- the arm
 SUM(is_churned) AS all_churn, -- every leaver
 SUM(churn_date >= '2025-05-01') -- left after the
 AS may_on -- offer began
FROM customer_features GROUP BY ab_group; -- two rows
""")

DatabaseError: Execution failed on sql '
SELECT ab_group, COUNT(*) AS n, -- the arm
 SUM(is_churned) AS all_churn, -- every leaver
 SUM(churn_date >= '2025-05-01') -- left after the
 AS may_on -- offer began
FROM customer_features GROUP BY ab_group; -- two rows
': no such table: customer_features

revenue, tested with the wrong measure and then the right one

In [11]:
("""
SELECT c.ab_group AS g, -- the arm
 COALESCE(SUM(r.amount_inr), 0) AS s -- paid, May-Jun
FROM customer_features c -- every sub
LEFT JOIN recharges r ON r.customer_id = c.customer_id -- keep all
 AND r.recharge_date >= '2025-05-01' -- the window
GROUP BY c.customer_id; -- one row per sub
""")

,g,s
0,B,0
1,A,0
2,A,1665
3,B,0
4,B,0
...,...,...
21995,A,3587
21996,A,0
21997,B,209
21998,A,0


prepaid against postpaid

In [12]:
q("""
SELECT connection_type AS grp, -- the split
 COUNT(*) AS n, SUM(is_churned) AS k -- size, leavers
FROM customer_features -- one row = a sub
GROUP BY connection_type; -- two rows
""")

,grp,n,k
0,Postpaid,4851,785
1,Prepaid,17149,4441


open ticket against never complained

In [13]:
q("""
SELECT CASE WHEN complaint_count = 0 THEN 'never' -- no ticket
 WHEN unresolved_count = 0 THEN 'all closed' -- all shut
 ELSE 'open' END AS grp, -- three groups
 COUNT(*) AS n, SUM(is_churned) AS k -- size, leavers
FROM customer_features GROUP BY grp; -- three rows
""")

"\nSELECT CASE WHEN complaint_count = 0 THEN 'never' -- no ticket\n WHEN unresolved_count = 0 THEN 'all closed' -- all shut\n ELSE 'open' END AS grp, -- three groups\n COUNT(*) AS n, SUM(is_churned) AS k -- size, leavers\nFROM customer_features GROUP BY grp; -- three rows\n"

first year against fourth year

In [14]:
q("""
SELECT tenure_years AS grp, -- 0 to 3
 COUNT(*) AS n, SUM(is_churned) AS k -- size, leavers
FROM customer_features GROUP BY tenure_years; -- four rows
""")

,grp,n,k
0,0,4610,1328
1,1,6961,1725
2,2,7055,1541
3,3,3374,632


a test that finds nothing

In [15]:
q("""
SELECT device_type AS grp, -- the split
 COUNT(*) AS n, SUM(is_churned) AS k -- size, leavers
FROM customer_features -- one row = a sub
GROUP BY device_type; -- two rows
""")

,grp,n,k
0,Feature Phone,3029,719
1,Smartphone,18971,4507


## The numbers — two queries, two grains

In [16]:
q("""
SELECT c.gender, c.device_type, -- two splits
 u.data_mb AS m -- one row per DAY
FROM usage_daily u -- one row = a day
JOIN customer_features c ON c.customer_id = u.customer_id; -- who
""")

,gender,device_type,m
0,Male,Smartphone,213.7
1,Male,Smartphone,1541.0
2,Male,Smartphone,107.4
3,Male,Smartphone,1569.6
4,Male,Smartphone,845.9
...,...,...,...
1326734,Male,Smartphone,335.9
1326735,Male,Smartphone,2465.5
1326736,Male,Smartphone,405.0
1326737,Male,Smartphone,1192.0


In [17]:
q("""
SELECT c.gender, c.device_type, -- two splits
 AVG(u.data_mb) AS m -- one row per SUB
FROM usage_daily u -- one row = a day
JOIN customer_features c ON c.customer_id = u.customer_id -- who
GROUP BY c.customer_id; -- one row per sub
""")

,gender,device_type,m
0,Male,Smartphone,729.705882
1,Male,Smartphone,256.590909
2,Female,Smartphone,1868.932727
3,Male,Feature Phone,218.880000
4,Female,Smartphone,1146.794872
...,...,...,...
21991,Male,Smartphone,6610.941772
21992,Male,Smartphone,1400.037624
21993,Female,Feature Phone,75.106154
21994,Female,Smartphone,1153.977778
